# 01 — From main sequence to white dwarf

We evolve a $1.2\,M_\odot$ star with SeBa, then check how much our requested timesteps change the result. Chapter 5 of the AMUSE book is useful here.

The route is **quick pilot → whole-track refinement → finer AGB tracks for the tidal tests**. 

**Important Note!** section 4 automatically reuses compatible saved work, preferring a batch with the requested long runs already done. Choose a folder explicitly to repeat a particular experiment. The final AGB runs can take up to around two hours from scratch (or much less if your pc is very good!). Set `RUN_FINE_AGB = False` to skip them.

In [ ]:
from pathlib import Path
from time import perf_counter
import csv
import json
import os
import platform
import sys
import uuid
import warnings
from contextlib import contextmanager
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version

import matplotlib.pyplot as plt
import numpy as np
from amuse.community.seba import Seba
from amuse.datamodel import Particle
from amuse.units import units


def locate_project_root(project_root=None):
    """Find the project from this notebook or a supplied folder. Super useful for actually finding the project."""
    if project_root is not None:
        candidate = Path(project_root).expanduser().resolve()
        if (candidate / "notebooks" / "01_stellar_evolution.ipynb").is_file():
            return candidate
        raise FileNotFoundError(f"Not a white_dwarf_project folder: {candidate}")

    cwd = Path.cwd().resolve()
    for parent in (cwd, *cwd.parents):
        for candidate in (parent, parent / "white_dwarf_project"):
            if (candidate / "notebooks" / "01_stellar_evolution.ipynb").is_file():
                return candidate
    raise FileNotFoundError(
        "Open this notebook from the cloned project, "
        "or set PROJECT_ROOT to your white_dwarf_project folder."
    )


PROJECT_ROOT = None  # Can be set manually to the path of the white_dwarf_project folder, but hopefully the code above finds it automatically.
WORKER_OPTIONS = {"channel_type": "sockets"}

ROOT = locate_project_root(PROJECT_ROOT)
DATA_DIR = ROOT / "data"
FIGURE_DIR = ROOT / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)


## 1. Choose a progenitor

We use $1.2\,M_\odot$ and $Z=0.02$ for this experiment. We request smaller steps during the giant phases because the mass and radius change faster there.

In [ ]:
M_INITIAL = 1.2 | units.MSun
METALLICITY = 0.02
MAX_AGE = 14000 | units.Myr
MAX_STEPS = 60000

# Useful stellar types: 3 = RGB, 5/6 = AGB, 10/11/12 = white dwarfs.
WHITE_DWARF_TYPES = {10, 11, 12}

# The numbers below will need to be adjusted if we change the stellar evolution code.
def next_output_step(stellar_type):
    """Pilot time sampling: finer output near giant and AGB phases since they evolve faster"""
    if stellar_type in (5, 6):
        return 0.01 | units.Myr  # AGB: 10 kyr
    if stellar_type in (2, 3):
        return 0.2 | units.Myr   # HG / RGB: 200 kyr
    if stellar_type == 4:
        return 0.5 | units.Myr   # Core He burning
    return 5 | units.Myr          # Main sequence


## 2. Run a quick pilot

This gives us a first look at the track and stops at the first saved white dwarf. The requested steps affect the calls to SeBa, not just how often we plot the result.

Should take about 10–30 seconds

In [ ]:
stellar = Seba(**WORKER_OPTIONS)
records = []
start = perf_counter()

try:
    stellar.parameters.metallicity = METALLICITY
    star = stellar.particles.add_particle(Particle(mass=M_INITIAL))

    def save_current_state():
        records.append({
            "time_Myr": float(stellar.model_time.value_in(units.Myr)),
            "mass_MSun": float(star.mass.value_in(units.MSun)),
            "radius_RSun": float(star.radius.value_in(units.RSun)),
            "luminosity_LSun": float(star.luminosity.value_in(units.LSun)),
            "stellar_type": int(star.stellar_type.value_in(units.stellar_type)),
        })

    save_current_state()
    for _ in range(MAX_STEPS):
        kind = records[-1]["stellar_type"]
        if kind in WHITE_DWARF_TYPES or stellar.model_time >= MAX_AGE:
            break

        target = stellar.model_time + next_output_step(kind)
        if target > MAX_AGE:
            target = MAX_AGE

        stellar.evolve_model(target)
        save_current_state()
    else:
        raise RuntimeError("MAX_STEPS reached before WD formation or MAX_AGE.")
finally:
    stellar.stop()

track = {key: np.array([row[key] for row in records]) for key in records[0]}
print(f"Saved {len(records):,} stellar states.")
print(f"Elapsed wall time: {perf_counter()-start:.2f} s")
print(f"Final time: {track['time_Myr'][-1]/1000:.3f} Gyr")
print(f"Final mass: {track['mass_MSun'][-1]:.3f} Msun")
print(f"Final stellar type: {track['stellar_type'][-1]}")
print("White dwarf reached:", track["stellar_type"][-1] in WHITE_DWARF_TYPES)


## 3. Look at the track and save it

The dashed line marks 1 au. The star crossing that radius does not by itself mean engulfment, mass loss also moves the planet outwards.

These are still coarse, sampled values. We check them with finer runs next.

In [ ]:
time_Gyr = track["time_Myr"] / 1000
mass = track["mass_MSun"]
radius = track["radius_RSun"]
luminosity = track["luminosity_LSun"]

fig, axes = plt.subplots(3, 1, figsize=(9, 9), sharex=True, constrained_layout=True)

axes[0].plot(time_Gyr, mass)
axes[0].set_ylabel(r"Mass [$M_\odot$]")

axes[1].plot(time_Gyr, radius)
axes[1].axhline(
    (1 | units.au).value_in(units.RSun),
    linestyle="--", linewidth=1, color="gray", label="1 au",
)
axes[1].set_yscale("log")
axes[1].set_ylabel(r"Radius [$R_\odot$]")
axes[1].legend(frameon=False)

axes[2].plot(time_Gyr, luminosity)
axes[2].set_yscale("log")
axes[2].set_ylabel(r"Luminosity [$L_\odot$]")
axes[2].set_xlabel("Time since ZAMS [Gyr]")

for ax in axes:
    ax.grid(alpha=0.2)

fig.suptitle(f"SeBa stellar evolution: initial mass = {M_INITIAL.value_in(units.MSun):g} Msun")
output_figure = FIGURE_DIR / "stellar_evolution_pilot.pdf"
fig.savefig(output_figure, bbox_inches="tight")
plt.show()
print("Figure saved:", output_figure)


In [ ]:
imax = int(np.argmax(radius))
print(f"Maximum sampled radius: {radius[imax]:.2f} Rsun")
print(f"                         {radius[imax] * (1 | units.RSun).value_in(units.au):.3f} au")
print(f"At time: {time_Gyr[imax]:.4f} Gyr")
print(f"Stellar type at max radius: {track['stellar_type'][imax]}")

wd_indices = np.flatnonzero(np.isin(track['stellar_type'], list(WHITE_DWARF_TYPES)))
if len(wd_indices):
    iwd = int(wd_indices[0])
    print(f"First sampled WD stage: {time_Gyr[iwd]:.4f} Gyr")
    print(f"White dwarf mass: {mass[iwd]:.4f} Msun")
else:
    print("No WD phase reached; try adjusting initial mass or MAX_AGE.")


### Keep the pilot for later examples (the more simple ones)

Notebook 02 reads data/stellar_track_pilot.npz, so we keep this file. If you don't have it, this is where its made.

In [ ]:
output_data = DATA_DIR / "stellar_track_pilot.npz"
np.savez_compressed(
    output_data,
    **track,
    initial_mass_MSun=M_INITIAL.value_in(units.MSun),
    metallicity=METALLICITY,
)
print("Saved stellar track:", output_data)


## 4. Refine the whole stellar track

The pilot had mass changes of nearly 10% between some saved states. We repeat the evolution with requested steps divided by 16, 32 and 64.

`STELLAR_RUN = None` finds a batch matching these stellar inputs, requested-step schedule and package versions. It prefers one with the requested fine AGB runs already done, then more saved uniform tracks, then the newest date. Missing runs are added, incomplete or incompatible saved runs are left for inspection. If nothing matches, we start a new batch.

Use `"new"` to start fresh, or a project root relative folder to choose a saved batch. The chosen folder is used for all later refinements.

The two switches below control the extra 128 and fine AGB runs. We use these in later notebooks so its good to have.

In [ ]:
# None: reuse the most complete matching batch, or start one if none exists.
# Use "new" for a fresh batch, or a project relative folder to choose one explicitly.
STELLAR_RUN = None
REFINEMENTS = (16, 32, 64)
RUN_EXTRA_REFINEMENT = True  # 128 run, needed before the fine AGB section.
EXTRA_REFINEMENT = 128
RUN_FINE_AGB = True  # Set False to skip the long 512/1024/2048 runs!

# These are working tolerances!
BOUNDARY_TOL_AU = 0.001
WD_MASS_TOL_MSUN = 1e-4
RADIUS_REL_TOL = 0.01
PLANET_MASS_MSUN = float((1.9 | units.MEarth).value_in(units.MSun))
BOUNDARY_ECCENTRICITY = 0.0  # A circular reference orbit.
RSUN_IN_AU = float((1 | units.RSun).value_in(units.au))

# Use the same requested-step schedule as the pilot before.
BASE_STEPS_MYR = {
    kind: float(next_output_step(kind).value_in(units.Myr))
    for kind in range(7)
}
TRACK_COLUMNS = (
    "time_Myr", "mass_MSun", "radius_RSun", "luminosity_LSun", "stellar_type",
)


def write_json(path, values):
    Path(path).write_text(
        json.dumps(values, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )


def write_csv(path, rows):
    with Path(path).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


@contextmanager
def worker_directory(path):
    previous = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous)


def installed_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "not available"


sampling_environment = {
    "python": sys.version,
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "packages": {
        name: installed_version(name)
        for name in ("amuse-framework", "amuse-seba", "numpy")
    },
    "worker_options": WORKER_OPTIONS,
}


def refinement_inputs(refinement):
    return {
        "initial_mass_MSun": float(M_INITIAL.value_in(units.MSun)),
        "metallicity": float(METALLICITY),
        "refinement": int(refinement),
        "base_steps_Myr": {str(k): v for k, v in BASE_STEPS_MYR.items()},
        "max_age_Myr": float(MAX_AGE.value_in(units.Myr)),
        "max_steps": int(MAX_STEPS * refinement),
        "packages": sampling_environment["packages"],
        "worker_options": dict(WORKER_OPTIONS),
    }


def same_stellar_inputs(saved, requested):
    def without_channel(values):
        values = dict(values)
        options = dict(values.get("worker_options", {}))
        options.pop("channel_type", None)
        values["worker_options"] = options
        return values

    return without_channel(saved) == without_channel(requested)


def choose_stellar_batch(selection=None):
    """Reuse matching saved work, make a new batch, or use an explicit folder. This function was hard to get right, but it should be future-proof."""
    sampling_root = (DATA_DIR / "stellar_sampling_runs").resolve()
    if selection is not None and selection != "new":
        relative = Path(selection)
        if relative.is_absolute():
            raise ValueError("Use a project-relative stellar batch path.")
        folder = (ROOT / relative).resolve()
        if folder.parent != sampling_root or not folder.is_dir():
            raise FileNotFoundError(
                f"Stellar batch not found: {folder}. "
                "Use STELLAR_RUN=None for automatic selection, or 'new' for a fresh batch."
            )
        return folder, "Explicit stellar batch"

    if selection is None:
        uniform_levels = set(REFINEMENTS)
        if RUN_EXTRA_REFINEMENT:
            uniform_levels.add(EXTRA_REFINEMENT)
        if RUN_FINE_AGB:
            uniform_levels.add(128)
        candidates = []

        def completed(run_dir, expected, mixed=False):
            if not run_dir.exists():
                return False
            info = json.loads((run_dir / "run.json").read_text(encoding="utf-8"))
            if not (
                isinstance(info, dict)
                and info.get("status") == "first_white_dwarf_reached"
                and (run_dir / "stellar_track.npz").is_file()
                and isinstance(info.get("inputs"), dict)
            ):
                raise ValueError("Incomplete saved run.")
            saved = info["inputs"].get("reference_inputs") if mixed else info["inputs"]
            if not isinstance(saved, dict) or not same_stellar_inputs(saved, expected):
                raise ValueError("Different stellar inputs or package versions.")
            return True

        for folder in sampling_root.glob("stellar_*"):
            if not folder.is_dir():
                continue
            try:
                uniform_done = sum(completed(
                    folder / f"refinement_{level:03d}", refinement_inputs(level),
                ) for level in sorted(uniform_levels))
                fine_done = sum(completed(folder / f"mixed_pre128_agb{level}", refinement_inputs(128), mixed=True)
                                for level in (512, 1024, 2048)) if RUN_FINE_AGB else 0
            except (OSError, ValueError, TypeError):
                continue
            if uniform_done:
                candidates.append((fine_done, uniform_done, folder.name, folder))
        if candidates:
            return max(candidates)[-1], "Reusing compatible stellar batch"
        print("No reusable batch matches these stellar inputs and package versions; starting fresh.")

    sampling_id = (
        datetime.now(timezone.utc).strftime("stellar_%Y%m%dT%H%M%SZ_")
        + uuid.uuid4().hex[:6]
    )
    folder = sampling_root / sampling_id
    folder.mkdir(parents=True, exist_ok=False)
    return folder, "New stellar batch"


# Pick once here, every later refinement uses this same folder.
SAMPLING_DIR, selection_note = choose_stellar_batch(STELLAR_RUN)
sampling_id = SAMPLING_DIR.name
SAMPLING_FIGURES = FIGURE_DIR / "stellar_sampling_runs" / sampling_id
SAMPLING_FIGURES.mkdir(parents=True, exist_ok=True)

if not (SAMPLING_DIR / "environment.json").exists():
    write_json(SAMPLING_DIR / "environment.json", sampling_environment)
print(selection_note + ":", SAMPLING_DIR.relative_to(ROOT).as_posix())
print("Refinement    MS step [Myr]    RGB step [yr]    AGB step [yr]")
for refinement in REFINEMENTS:
    print(
        f"{refinement:10d} {BASE_STEPS_MYR[1]/refinement:16.5f} "
        f"{1e6*BASE_STEPS_MYR[3]/refinement:16.2f} "
        f"{1e6*BASE_STEPS_MYR[5]/refinement:16.2f}"
    )


### Save and reuse each run

This starts a fresh SeBa worker for each refinement. 

In [ ]:
def validate_stellar_track(values):
    arrays = [np.asarray(values[key]) for key in TRACK_COLUMNS]
    if arrays[0].ndim != 1 or len(arrays[0]) < 2 or any(a.shape != arrays[0].shape for a in arrays):
        raise ValueError("Stellar columns must be equal-length 1D arrays.")
    if not all(np.isfinite(a).all() for a in arrays):
        raise ValueError("Non-finite stellar output.")
    if arrays[0][0] != 0 or not np.all(np.diff(arrays[0]) > 0):
        raise ValueError("Require increasing times starting at ZAMS.")
    if np.any(arrays[1] <= 0) or np.any(arrays[2] <= 0):
        raise ValueError("Mass and radius must be positive.")
    if not np.equal(arrays[4], np.floor(arrays[4])).all():
        raise ValueError("Invalid stellar-type values.")
    if arrays[4][-1] not in WHITE_DWARF_TYPES:
        raise ValueError("This run did not reach a white dwarf.")
    if np.any(np.isin(arrays[4][:-1], list(WHITE_DWARF_TYPES))):
        raise ValueError("Expected a track ending at its first saved WD state.")


def pack_stellar_states(records, refinement):
    values = np.asarray(records)
    result = {key: values[:, j].copy() for j, key in enumerate(TRACK_COLUMNS)}
    result["stellar_type"] = result["stellar_type"].astype(np.int64)
    result.update(
        initial_mass_MSun=np.asarray(float(M_INITIAL.value_in(units.MSun))),
        metallicity=np.asarray(float(METALLICITY)),
        request_refinement=np.asarray(refinement),
    )
    return result





def run_refined_track(refinement, batch_dir=None):
    if refinement < 1 or int(refinement) != refinement:
        raise ValueError("Refinement must be a positive integer.")
    refinement = int(refinement)
    batch_dir = SAMPLING_DIR if batch_dir is None else Path(batch_dir).resolve()
    folder = batch_dir / f"refinement_{refinement:03d}"
    path = folder / "stellar_track.npz"
    info_path = folder / "run.json"
    inputs = refinement_inputs(refinement)

    if folder.exists():
        if path.is_file() and info_path.is_file():
            info = json.loads(info_path.read_text(encoding="utf-8"))
            completed = info.get("status") == "first_white_dwarf_reached"
            if completed and same_stellar_inputs(info.get("inputs", {}), inputs):
                with np.load(path, allow_pickle=False) as saved:
                    result = {key: saved[key].copy() for key in saved.files}
                validate_stellar_track(result)
                expected = {
                    "initial_mass_MSun": inputs["initial_mass_MSun"],
                    "metallicity": inputs["metallicity"],
                    "request_refinement": refinement,
                }
                if any(
                    key not in result or np.asarray(result[key]).shape != ()
                    or float(result[key]) != value
                    for key, value in expected.items()
                ) or len(result["time_Myr"]) != info.get("n_states"):
                    raise ValueError(f"Track and saved run settings disagree: {path}")
                print(f"Reusing completed refinement {refinement}.", flush=True)
                return result, info["wall_seconds"], path
        raise RuntimeError(
            f"{folder} contains an incomplete or different run. "
            "Use a new batch or restore the settings that produced this track."
        )

    folder.mkdir()
    write_json(info_path, {"status": "started", "inputs": inputs})
    records, stellar = [], None
    start = last_report = perf_counter()
    print(f"Starting refinement {refinement} ...", flush=True)
    with worker_directory(folder):
        try:
            stellar = Seba(**WORKER_OPTIONS)
            stellar.parameters.metallicity = METALLICITY
            star = stellar.particles.add_particle(Particle(mass=M_INITIAL))

            def record_state():
                records.append((
                    float(stellar.model_time.value_in(units.Myr)),
                    float(star.mass.value_in(units.MSun)),
                    float(star.radius.value_in(units.RSun)),
                    float(star.luminosity.value_in(units.LSun)),
                    int(star.stellar_type.value_in(units.stellar_type)),
                ))

            record_state()
            for step in range(inputs["max_steps"]):
                age, _, _, _, kind = records[-1]
                if kind in WHITE_DWARF_TYPES:
                    break
                if age >= inputs["max_age_Myr"]:
                    raise RuntimeError("MAX_AGE reached without a white dwarf.")
                if kind not in BASE_STEPS_MYR:
                    raise ValueError(f"No requested step defined for stellar type {kind}.")
                target = min(age + BASE_STEPS_MYR[kind]/refinement, inputs["max_age_Myr"])
                stellar.evolve_model(target | units.Myr)
                record_state()
                if records[-1][0] <= age:
                    raise RuntimeError("SeBa did not advance its time.")
                if (step+1) % 20000 == 0 or perf_counter()-last_report >= 30:
                    print(f"  {len(records):,} states; age = {records[-1][0]/1000:.6f} Gyr; "
                          f"elapsed = {(perf_counter()-start)/60:.1f} min", flush=True)
                    last_report = perf_counter()
            if records[-1][-1] not in WHITE_DWARF_TYPES:
                raise RuntimeError("Step or age limit reached before WD formation.")
            result = pack_stellar_states(records, refinement)
            validate_stellar_track(result)
        except BaseException as exc:
            if records:
                np.savez_compressed(folder / "incomplete_track.npz", **pack_stellar_states(records, refinement))
            write_json(info_path, {"status": "failed", "inputs": inputs,
                                  "saved_states": len(records), "exception": repr(exc)})
            raise
        finally:
            if stellar is not None:
                try:
                    stellar.stop()
                except Exception as exc:
                    warnings.warn(f"Could not stop SeBa cleanly: {exc!r}")
    elapsed = perf_counter() - start
    np.savez_compressed(path, **result)
    write_json(info_path, {
        "status": "first_white_dwarf_reached", "inputs": inputs,
        "n_states": len(records), "wall_seconds": elapsed,
    })
    print(f"  Finished: {len(records):,} states in {elapsed:.1f} s", flush=True)
    return result, elapsed, path


### Run 16, 32 and 64

Roughly 17–35 minutes (very dependent on your specs). Loading matching saved runs is much faster.

In [ ]:
if len(REFINEMENTS) < 3 or any(int(r) != r or r < 1 for r in REFINEMENTS):
    raise ValueError("Choose at least three positive integer refinements.")
if any(b != 2*a for a, b in zip(REFINEMENTS[:-1], REFINEMENTS[1:])):
    raise ValueError("Use successive doublings, e.g. (16, 32, 64).")

refined_tracks, refinement_times, refinement_paths = {}, {}, {}
for refinement in REFINEMENTS:
    result, elapsed, path = run_refined_track(refinement)
    refined_tracks[refinement] = result
    refinement_times[refinement] = elapsed
    refinement_paths[refinement] = path


write_json(SAMPLING_DIR / "settings.json", {
    "initial_mass_MSun": float(M_INITIAL.value_in(units.MSun)),
    "metallicity": float(METALLICITY), "refinements": list(REFINEMENTS),
    "base_steps_Myr": BASE_STEPS_MYR, "planet_mass_MSun": PLANET_MASS_MSUN,
    "boundary_eccentricity": BOUNDARY_ECCENTRICITY,
    "boundary_tolerance_AU": BOUNDARY_TOL_AU,
    "WD_mass_tolerance_MSun": WD_MASS_TOL_MSUN,
    "radius_relative_tolerance": RADIUS_REL_TOL,
})


## 5. Have the main quantities settled?

We compare the first-WD mass, the largest sampled RGB/AGB radii, and a simple geometric boundary. With slow isotropic mass loss and fixed eccentricity,

$$a(t)=a_0\frac{M_{\star,0}+m_p}{M_\star(t)+m_p},\qquad q(t)=a(t)(1-e_0).$$

Setting $q=R_\star$ gives the initial distance needed to avoid radius crossing in this model:

$$a_{0,\mathrm{crit}}=\max_t\left[R_\star(t)\frac{M_\star(t)+m_p}{(M_{\star,0}+m_p)(1-e_0)}\right].$$

This treats the planet as a point and leaves out tides and the brown dwarf, just a geometric check.

In [ ]:
def summarize_stellar_track(values, label, refinement=None, elapsed=None):
    validate_stellar_track(values)
    age = values["time_Myr"]
    mass = values["mass_MSun"]
    kind = values["stellar_type"]
    radius_AU = values["radius_RSun"] * RSUN_IN_AU
    if not 0 <= BOUNDARY_ECCENTRICITY < 1:
        raise ValueError("The initial eccentricity must be between 0 and 1.")
    critical = radius_AU * (mass + PLANET_MASS_MSUN) / (
        (mass[0] + PLANET_MASS_MSUN) * (1 - BOUNDARY_ECCENTRICITY)
    )
    imax = int(np.argmax(critical))
    fraction = abs(np.diff(mass)) / mass[:-1]
    imass = int(np.argmax(fraction))

    def phase_max(array, types):
        selected = np.isin(kind, types)
        return float(array[selected].max()) if selected.any() else None

    return {
        "run": label, "refinement": refinement, "n_states": len(age), "wall_seconds": elapsed,
        "first_WD_time_Myr": float(age[-1]),
        "WD_transition_bracket_yr": float(1e6*(age[-1]-age[-2])),
        "mass_WD_MSun": float(mass[-1]),
        "max_radius_RGB_AU": phase_max(radius_AU, [3]),
        "max_radius_AGB_AU": phase_max(radius_AU, [5, 6]),
        "a_critical_RGB_AU": phase_max(critical, [3]),
        "a_critical_AGB_AU": phase_max(critical, [5, 6]),
        "a_critical_all_AU": float(critical[imax]),
        "critical_time_Myr": float(age[imax]), "critical_stellar_type": int(kind[imax]),
        "max_adjacent_mass_fraction": float(fraction[imass]),
        "largest_mass_step_start_Myr": float(age[imass]),
        "largest_mass_step_end_Myr": float(age[imass+1]),
    }


sampling_summary = []
pilot_path = DATA_DIR / "stellar_track_pilot.npz"
if pilot_path.is_file():
    with np.load(pilot_path, allow_pickle=False) as saved:
        pilot_track = {key: saved[key].copy() for key in saved.files}
    same_inputs = (
        "initial_mass_MSun" in pilot_track and "metallicity" in pilot_track
        and np.isclose(float(pilot_track["initial_mass_MSun"]),
                       float(M_INITIAL.value_in(units.MSun)), rtol=1e-12, atol=0)
        and np.isclose(float(pilot_track["metallicity"]), METALLICITY, rtol=1e-12, atol=0)
    )
    if same_inputs:
        # The old pilot is just a reference here
        sampling_summary.append(summarize_stellar_track(pilot_track, "original_pilot"))
    else:
        print("The saved pilot has different or missing input settings. leaving it out of this comparison.")

for refinement in REFINEMENTS:
    sampling_summary.append(summarize_stellar_track(
        refined_tracks[refinement], f"refinement_{refinement}",
        refinement, refinement_times[refinement],
    ))
write_csv(SAMPLING_DIR / "stellar_convergence_summary.csv", sampling_summary)

print(f"{'Run':<18} {'States':>8} {'M_WD':>10} {'R_RGB':>10} {'R_AGB':>10} "
      f"{'acrit_RGB':>11} {'acrit_AGB':>11} {'max dM/M':>11}")
for row in sampling_summary:
    show = lambda key: f"{row[key]:.6f}" if row[key] is not None else "missing"
    print(f"{row['run']:<18} {row['n_states']:8d} {show('mass_WD_MSun'):>10} "
          f"{show('max_radius_RGB_AU'):>10} {show('max_radius_AGB_AU'):>10} "
          f"{show('a_critical_RGB_AU'):>11} {show('a_critical_AGB_AU'):>11} "
          f"{100*row['max_adjacent_mass_fraction']:10.4f}%")
print("Masses are in Msun; radii and initial critical distances are in au.")

### Compare neighbouring refinements

Our working limits are 0.001 au for the boundaries, 1% for the radii, and $10^{-4}\,M_\odot$ for the WD mass. We require the last two comparisons to pass. 

In [ ]:
def compare_stellar_summaries(coarse, fine):
    row = {"coarse": coarse["refinement"], "fine": fine["refinement"]}
    boundary_ok = radius_ok = True
    for key in ("a_critical_all_AU", "a_critical_RGB_AU", "a_critical_AGB_AU"):
        delta = abs(fine[key]-coarse[key]) if coarse[key] is not None and fine[key] is not None else None
        row["delta_" + key] = delta
        boundary_ok = boundary_ok and delta is not None and delta <= BOUNDARY_TOL_AU
    for key in ("max_radius_RGB_AU", "max_radius_AGB_AU"):
        delta = abs(fine[key]-coarse[key])/fine[key] if coarse[key] is not None and fine[key] not in (None, 0) else None
        row["relative_delta_" + key] = delta
        radius_ok = radius_ok and delta is not None and delta <= RADIUS_REL_TOL
    row["delta_mass_WD_MSun"] = abs(fine["mass_WD_MSun"]-coarse["mass_WD_MSun"])
    row["delta_first_WD_time_yr"] = 1e6*abs(fine["first_WD_time_Myr"]-coarse["first_WD_time_Myr"])
    row["boundaries_pass"] = bool(boundary_ok)
    row["radii_pass"] = bool(radius_ok)
    row["WD_mass_pass"] = row["delta_mass_WD_MSun"] <= WD_MASS_TOL_MSUN
    row["pass_working_tolerances"] = all(row[key] for key in ("boundaries_pass", "radii_pass", "WD_mass_pass"))
    return row


by_refinement = {row["refinement"]: row for row in sampling_summary if row["refinement"] is not None}
sampling_comparisons = [
    compare_stellar_summaries(by_refinement[coarse], by_refinement[fine])
    for coarse, fine in zip(REFINEMENTS[:-1], REFINEMENTS[1:])
]
write_csv(SAMPLING_DIR / "stellar_refinement_comparisons.csv", sampling_comparisons)
converged_for_test = len(sampling_comparisons) >= 2 and all(
    row["pass_working_tolerances"] for row in sampling_comparisons[-2:]
)
for row in sampling_comparisons:
    print(f"{row['coarse']} -> {row['fine']}: boundaries={row['boundaries_pass']}, "
          f"radii={row['radii_pass']}, WD mass={row['WD_mass_pass']}")
print("Overall:", "PASS for these working tolerances" if converged_for_test else "REVIEW / REFINE FURTHER")

In [ ]:
# Plot the quantities used in the comparison. Some of these plots are redundant but are good for diagnostics.
levels = np.array(REFINEMENTS)
fig, axes = plt.subplots(3, 1, figsize=(9, 9), sharex=True, constrained_layout=True)
for phase in ("RGB", "AGB"):
    axes[0].plot(levels, [by_refinement[r][f"max_radius_{phase}_AU"] for r in levels], "o-", label=phase)
    axes[1].plot(levels, [by_refinement[r][f"a_critical_{phase}_AU"] for r in levels], "o-", label=phase)
axes[2].plot(levels, [by_refinement[r]["mass_WD_MSun"] for r in levels], "o-")
axes[0].set_ylabel("Maximum sampled radius [au]")
axes[1].set_ylabel("Critical initial semimajor axis [au]")
axes[2].set_ylabel(r"WD mass [$M_\odot$]")
axes[2].set_xlabel("Refinement factor")
axes[2].set_xscale("log", base=2)
axes[2].set_xticks(levels, [str(r) for r in levels])
for ax in axes:
    ax.grid(alpha=0.2)
for ax in axes[:2]:
    ax.legend(frameon=False)
fig.suptitle("SeBa requested-step comparison")
fig.savefig(SAMPLING_FIGURES / "stellar_sampling_convergence.pdf", bbox_inches="tight")
plt.show()


## 6. Save the track choice

We save the chosen track and its summary checks. Notebook 04 can find an eligible batch automatically once 32/64/128 are available.

In [ ]:
finest = max(REFINEMENTS)
finest_path = refinement_paths[finest]
finest_relative = finest_path.relative_to(ROOT).as_posix()
sampling_status = {
    "status": (
        "passed_working_tolerances"
        if converged_for_test else "needs_further_refinement"
    ),
    "passed_working_tolerances": bool(converged_for_test),
    "finest_refinement": int(finest),
    "finest_candidate_track": finest_relative,
    "selected_track": finest_relative if converged_for_test else None,
    "last_two_comparisons": sampling_comparisons[-2:],
    "finest_summary": by_refinement[finest],
    "scope": (
        "Requested-step convergence of the first-WD mass "
        "and sampled radii/geometric boundaries."
    ),
    "not_proof_of_physical_accuracy": True,
}
write_json(SAMPLING_DIR / "validation_status.json", sampling_status)
print(
    "Selected track:" if converged_for_test else "Finest candidate (provisional):",
    finest_relative,
)
if converged_for_test:
    print("\nFor the input settings in notebook 04:")
    print(f'STELLAR_RUN = "{SAMPLING_DIR.relative_to(ROOT).as_posix()}"')
else:
    print("The finest track is saved, but has not passed these checks.")
print("\nFiles to inspect:")
for name in (
    "stellar_convergence_summary.csv",
    "stellar_refinement_comparisons.csv",
    "validation_status.json",
):
    print((SAMPLING_DIR / name).relative_to(ROOT).as_posix())


## 7. Add uniform refinement 128

Notebook 04 uses this for its next stellar comparison, and the fine AGB section below uses it as a reference. `RUN_EXTRA_REFINEMENT` controls this step. A matching saved 128 track is reused.

Took about 20 mins.

In [ ]:
if RUN_EXTRA_REFINEMENT:
    if (
        EXTRA_REFINEMENT < 2 or int(EXTRA_REFINEMENT) != EXTRA_REFINEMENT
        or EXTRA_REFINEMENT % 2
    ):
        raise ValueError("Choose an even positive integer refinement.")
    extra_level = int(EXTRA_REFINEMENT)
    extra_batch = SAMPLING_DIR

    # Check against the previous level 
    previous = extra_level // 2
    previous_folder = extra_batch / f"refinement_{previous:03d}"
    previous_info = previous_folder / "run.json"
    if not previous_info.is_file() or not (previous_folder / "stellar_track.npz").is_file():
        raise FileNotFoundError(f"Refinement {previous} is missing from {extra_batch}")
    saved_info = json.loads(previous_info.read_text(encoding="utf-8"))
    if (
        saved_info.get("status") != "first_white_dwarf_reached"
        or not same_stellar_inputs(
            saved_info.get("inputs", {}), refinement_inputs(previous),
        )
    ):
        raise ValueError(
            "The previous track is incomplete or its inputs/package versions differ. "
            "Use matching settings or make a new batch."
        )

    extra_track, extra_elapsed, extra_path = run_refined_track(
        extra_level, batch_dir=extra_batch,
    )
    extra_summary = summarize_stellar_track(
        extra_track, f"refinement_{extra_level}", extra_level, extra_elapsed,
    )
    write_json(extra_path.parent / "summary.json", extra_summary)
    print("Additional track:", extra_path.relative_to(ROOT).as_posix())
    print(f"Saved states: {extra_summary['n_states']:,}")
    print(f"First-WD mass: {extra_summary['mass_WD_MSun']:.12f} MSun")
    print(
        'STELLAR_RUN = '
        f'"{extra_batch.relative_to(ROOT).as_posix()}"'
    )
else:
    print("Additional refinement skipped.")


## 8. Resolve the late AGB changes with better refinement

The tidal pilot showed that the shrinking stellar radius needs finer sampling. We keep the earlier evolution at refinement 128 and use finer requests after its first saved AGB state. Note that these are not full tracks of these refinements!!!

Each new run reproduces and checks the earlier 128 track before starting. 

In [ ]:
# Keep the earlier evolution fixed and refine the requests after the first AGB state.
def run_agb_refinement(reference_path, subdivisions=2):
    reference_path = Path(reference_path).resolve()
    if not reference_path.is_relative_to(ROOT.resolve()):
        raise ValueError("Choose a reference track inside this project.")
    if subdivisions not in (2, 4, 8, 16):
        raise ValueError("Use subdivisions=2, 4, 8 or 16 for AGB 256, 512, 1024 or 2048.")
    post_refinement = 128*int(subdivisions)

    info = json.loads((reference_path.parent / "run.json").read_text())
    inputs = info["inputs"]
    with np.load(reference_path, allow_pickle=False) as data:
        reference = {key: data[key].copy() for key in data.files}
    validate_stellar_track(reference)
    if info.get("status") != "first_white_dwarf_reached":
        raise ValueError("The reference stellar run is incomplete.")
    if (inputs["refinement"] != 128 or np.asarray(reference["request_refinement"]).shape != ()
        or float(reference["request_refinement"]) != 128):
        raise ValueError("Use the completed uniform refinement-128 track.")
    if len(reference["time_Myr"]) != info["n_states"]:
        raise ValueError("The reference track and run record disagree.")
    for key in ("initial_mass_MSun", "metallicity"):
        if float(reference[key]) != inputs[key]:
            raise ValueError(f"The reference metadata disagree for {key}.")

    agb = np.flatnonzero(np.isin(reference["stellar_type"], (5, 6)))
    if not len(agb):
        raise ValueError("The reference contains no AGB states.")
    branch = int(agb[0])
    base_steps = {int(k): float(v) for k, v in inputs["base_steps_Myr"].items()}
    if any(not np.isfinite(v) or v <= 0 for v in base_steps.values()):
        raise ValueError("Invalid reference requested steps.")
    options = {**inputs.get("worker_options", {}), "channel_type": "sockets"}
    folder = reference_path.parent.parent / f"mixed_pre128_agb{post_refinement}"
    path = folder / "stellar_track.npz"
    settings = {
        "schedule_kind": "mixed_requested_steps",
        "reference_track": reference_path.relative_to(ROOT).as_posix(),
        "reference_inputs": inputs,
        "reference_states": len(reference["time_Myr"]),
        "branch_reference_index": branch,
        "branch_age_Myr": float(reference["time_Myr"][branch]),
        "pre_agb_refinement": 128,
        "post_first_saved_agb_refinement": post_refinement,
        "prefix_age_atol_Myr": 1e-9,
        "prefix_state_rtol": 1e-9,
        "prefix_state_atol": 1e-12,
        "requested_age_atol_Myr": 1e-9,
        "worker_options": options,
    }
    prefix_errors = {key: 0.0 for key in ("age_Myr", "mass_relative", "radius_relative", "luminosity_relative")}

    def check_prefix(values, j):
        expected = np.array([reference[key][j] for key in TRACK_COLUMNS], dtype=float)
        values = np.asarray(values, dtype=float)
        prefix_errors["age_Myr"] = max(prefix_errors["age_Myr"], abs(float(values[0]-expected[0])))
        for k, name in enumerate(("mass_relative", "radius_relative", "luminosity_relative"), 1):
            error = abs(float(values[k]-expected[k])) / max(abs(float(expected[k])), 1e-12)
            prefix_errors[name] = max(prefix_errors[name], error)
        if (abs(values[0]-expected[0]) > settings["prefix_age_atol_Myr"]
            or not np.allclose(values[1:4], expected[1:4], rtol=settings["prefix_state_rtol"],
                               atol=settings["prefix_state_atol"])
            or values[4] != expected[4]):
            raise RuntimeError(f"The reference prefix was not reproduced at state {j}. "
                               "The experiment has stopped before claiming a shared earlier evolution.")

    def validate_mixed(result):
        validate_stellar_track(result)
        size = len(result["time_Myr"])
        if size <= branch+1 or "request_refinement" in result:
            raise ValueError("Invalid mixed-schedule track.")
        for key, value in (("initial_mass_MSun", inputs["initial_mass_MSun"]),
                           ("metallicity", inputs["metallicity"]),
                           ("pre_agb_refinement", 128), ("post_first_saved_agb_refinement", post_refinement)):
            if np.asarray(result[key]).shape != () or float(result[key]) != value:
                raise ValueError(f"Mixed-track metadata disagree for {key}.")
        if str(result["schedule_kind"]) != "mixed_requested_steps":
            raise ValueError("The saved schedule label is incorrect.")
        wanted_levels = np.r_[0, np.full(branch, 128), np.full(size-branch-1, post_refinement)]
        if not np.array_equal(result["request_refinement_used"], wanted_levels):
            raise ValueError("The recorded refinement schedule is incorrect.")
        wanted_targets = np.zeros(size)
        for j in range(1, size):
            source = reference if j <= branch else result
            wanted_targets[j] = min(float(source["time_Myr"][j-1])
                + base_steps[int(source["stellar_type"][j-1])]/wanted_levels[j], inputs["max_age_Myr"])
        targets = np.asarray(result["requested_time_Myr"])
        if targets.shape != (size,) or not np.isfinite(targets).all() or not np.array_equal(targets, wanted_targets):
            raise ValueError("The saved requested ages disagree with the schedule.")
        if np.max(abs(targets-result["time_Myr"])) > settings["requested_age_atol_Myr"]:
            raise ValueError("Returned stellar ages differ from the requested ages.")

    if folder.exists():
        previous_file = folder / "run.json"
        previous = json.loads(previous_file.read_text()) if previous_file.is_file() else {}
        if previous.get("status") != "first_white_dwarf_reached" or previous.get("inputs") != settings or not path.is_file():
            raise RuntimeError(f"An incomplete or different experiment exists in {folder}. Keep it for inspection.")
        with np.load(path, allow_pickle=False) as data:
            result = {key: data[key].copy() for key in data.files}
        validate_mixed(result)
        if len(result["time_Myr"]) != previous["n_states"]:
            raise ValueError("The saved mixed track and metadata disagree.")
        for j in range(branch+1):
            check_prefix([result[key][j] for key in TRACK_COLUMNS], j)
        print("Reusing completed experiment:", folder.relative_to(ROOT).as_posix())
        return reference, result, folder

    for package, saved_version in inputs["packages"].items():
        if installed_version(package) != saved_version:
            raise ValueError(f"Use the reference environment: {package}={saved_version}.")
    folder.mkdir()
    write_json(folder / "run.json", {"status": "started", "inputs": settings})
    records, requested, levels = [], [], []
    stellar = None
    started = last_print = perf_counter()

    def pack():
        array = np.asarray(records, dtype=float)
        result = {key: array[:, j].copy() for j, key in enumerate(TRACK_COLUMNS)}
        result["stellar_type"] = result["stellar_type"].astype(np.int64)
        result.update(
            requested_time_Myr=np.asarray(requested),
            request_refinement_used=np.asarray(levels, dtype=np.int64),
            initial_mass_MSun=np.asarray(inputs["initial_mass_MSun"]),
            metallicity=np.asarray(inputs["metallicity"]),
            schedule_kind=np.asarray("mixed_requested_steps"),
            pre_agb_refinement=np.asarray(128),
            post_first_saved_agb_refinement=np.asarray(post_refinement),
        )
        return result

    print(f"Reproducing {branch+1:,} states through the first saved AGB state...", flush=True)
    with worker_directory(folder):
        try:
            stellar = Seba(**options)
            stellar.parameters.metallicity = inputs["metallicity"]
            star = stellar.particles.add_particle(Particle(mass=inputs["initial_mass_MSun"] | units.MSun))

            def record(target, refinement):
                row = (
                    float(stellar.model_time.value_in(units.Myr)),
                    float(star.mass.value_in(units.MSun)),
                    float(star.radius.value_in(units.RSun)),
                    float(star.luminosity.value_in(units.LSun)),
                    int(star.stellar_type.value_in(units.stellar_type)),
                )
                if not np.isfinite(row).all() or row[1] <= 0 or row[2] <= 0:
                    raise RuntimeError("SeBa returned an invalid state.")
                if records and row[0] <= records[-1][0]:
                    raise RuntimeError("SeBa did not advance its time.")
                records.append(row)
                requested.append(target)
                levels.append(refinement)  # Zero marks the initial state, before any request.
                if abs(row[0]-target) > settings["requested_age_atol_Myr"]:
                    raise RuntimeError("SeBa did not reach the requested age.")

            record(0.0, 0)
            check_prefix(records[-1], 0)
            for step in range(int(subdivisions)*int(inputs["max_steps"])):
                j = len(records)
                age, _, _, _, kind = records[-1]
                if kind in WHITE_DWARF_TYPES:
                    break
                if age >= inputs["max_age_Myr"]:
                    raise RuntimeError("The maximum age was reached before WD formation.")
                if j <= branch:
                    # Reconstruct the original request, rather than requesting its returned age.
                    previous_age = float(reference["time_Myr"][j-1])
                    previous_kind = int(reference["stellar_type"][j-1])
                    target = min(previous_age + base_steps[previous_kind]/128, inputs["max_age_Myr"])
                    refinement = 128
                else:
                    target = min(age + base_steps[kind]/post_refinement, inputs["max_age_Myr"])
                    refinement = post_refinement
                stellar.evolve_model(target | units.Myr)
                record(target, refinement)
                if j <= branch:
                    check_prefix(records[-1], j)
                if j == branch:
                    print(f"Earlier evolution reproduced; using refinement {post_refinement} from here.", flush=True)
                if perf_counter()-last_print >= 30:
                    print(f"  {len(records):,} states; age={records[-1][0]/1000:.6f} Gyr; "
                          f"elapsed={(perf_counter()-started)/60:.1f} min", flush=True)
                    last_print = perf_counter()
            if records[-1][-1] not in WHITE_DWARF_TYPES:
                raise RuntimeError("The step limit was reached before WD formation.")
            result = pack()
            validate_mixed(result)
        except BaseException as exc:
            if records:
                np.savez_compressed(folder / "incomplete_track.npz", **pack())
            write_json(folder / "run.json", {
                "status": "failed", "inputs": settings, "exception": repr(exc),
                "saved_states": len(records), "prefix_errors": prefix_errors,
            })
            raise
        finally:
            if stellar is not None:
                try:
                    stellar.stop()
                except Exception as exc:
                    warnings.warn(f"Could not stop SeBa cleanly: {exc!r}")

    np.savez_compressed(path, **result)
    write_json(folder / "run.json", {
        "status": "first_white_dwarf_reached", "inputs": settings,
        "n_states": len(records), "wall_seconds": perf_counter()-started,
        "prefix_errors": prefix_errors,
        "max_requested_age_error_Myr": float(np.max(abs(result["time_Myr"]-result["requested_time_Myr"]))),
        "WD_transition_bracket_Myr": [float(result["time_Myr"][-2]), float(result["time_Myr"][-1])],
    })
    write_json(folder / "summary.json", {
        "run": folder.name, "schedule_kind": "mixed_requested_steps",
        "n_states": len(records), "prefix_reproduced": True,
        "first_saved_AGB_age_Myr": float(result["time_Myr"][branch]),
        "last_pre_WD_age_Myr": float(result["time_Myr"][-2]),
        "first_saved_WD_age_Myr": float(result["time_Myr"][-1]),
        "first_saved_WD_mass_MSun": float(result["mass_MSun"][-1]),
        "first_saved_WD_radius_RSun": float(result["radius_RSun"][-1]),
    })
    print("Saved:", path.relative_to(ROOT).as_posix())
    print(f"Completed {len(records):,} states in {(perf_counter()-started)/60:.1f} min.")
    return reference, result, folder

### Run the fine AGB tracks

**This is the long part.** The saved 1024 and 2048 runs took about 35 minutes. I already hhad 512 saved but if you don't have it allow more than an hour for all of them.

In [ ]:
# Reuse matching completed tracks; otherwise each call starts a new SeBa worker.
if RUN_FINE_AGB:
    AGB_REFERENCE_PATH = SAMPLING_DIR / "refinement_128" / "stellar_track.npz"
    if not AGB_REFERENCE_PATH.is_file():
        raise FileNotFoundError("Run uniform refinement 128 in this batch first.")
    agb_reference_path = AGB_REFERENCE_PATH
    agb_reference, agb_mixed_512, agb_folder_512 = run_agb_refinement(agb_reference_path, subdivisions=4)
    _, agb_mixed_1024, agb_folder_1024 = run_agb_refinement(agb_reference_path, subdivisions=8)
    _, agb_mixed_2048, agb_folder_2048 = run_agb_refinement(agb_reference_path, subdivisions=16)
else:
    print("Fine AGB runs skipped.")
